# BÀI TẬP THỰC HÀNH LẬP TRÌNH PYTHON
**Môn học:** Phân tích Dữ liệu và Học sâu  
**Sinh viên:** Nguyễn Quốc Trung  
**MSSV:** 2474802010412

Bài làm sử dụng dữ liệu tự tạo bằng `numpy`, sau đó xử lý bằng `pandas` và xây dựng mô hình CNN bằng `TensorFlow/Keras`.


## PHẦN 1: KHỞI TẠO VÀ XỬ LÝ DỮ LIỆU

### Câu 1: Khởi tạo dữ liệu

Tạo dữ liệu gồm 500 khách hàng. Dùng `np.random.seed()` để khi chạy lại vẫn cho kết quả giống nhau.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

so_luong = 500

# Tạo các cột dữ liệu
ma_kh = [f"KH{i:03d}" for i in range(1, so_luong + 1)]
tuoi = np.random.randint(18, 71, so_luong).astype(float)
thu_nhap = np.random.uniform(5_000_000, 50_000_000, so_luong)
gioi_tinh = np.random.choice(["Nam", "Nữ"], so_luong).astype(object)
thanh_pho = np.random.choice(["Hà Nội", "Đà Nẵng", "TP.HCM"], so_luong)

# Tổng chi tiêu có tương quan nhẹ với thu nhập
tong_chi_tieu = thu_nhap * np.random.uniform(0.25, 0.55, so_luong)
tong_chi_tieu += np.random.normal(0, 1_500_000, so_luong)
tong_chi_tieu = np.maximum(tong_chi_tieu, 0)

# Chèn 10 giá trị NaN vào cột Tuổi
vi_tri_nan_tuoi = np.random.choice(so_luong, 10, replace=False)
tuoi[vi_tri_nan_tuoi] = np.nan

# Chèn 15 giá trị NaN vào cột Giới tính
vi_tri_nan_gt = np.random.choice(so_luong, 15, replace=False)
gioi_tinh[vi_tri_nan_gt] = np.nan

# Tạo 5 giá trị thu nhập cực trị
vi_tri_outlier = np.random.choice(so_luong, 5, replace=False)
thu_nhap[vi_tri_outlier] = np.random.uniform(
    100_000_000, 200_000_000, 5
)

df_khachhang = pd.DataFrame({
    "MaKH": ma_kh,
    "Tuoi": tuoi,
    "ThuNhap": thu_nhap,
    "GioiTinh": gioi_tinh,
    "ThanhPho": thanh_pho,
    "TongChiTieu": tong_chi_tieu
})

df_khachhang.head()


### Câu 2: Xử lý giá trị khuyết

In [ ]:
# Kiểm tra số lượng giá trị khuyết trước khi xử lý
print("Số lượng giá trị khuyết ban đầu:")
print(df_khachhang.isnull().sum())

# Điền Tuổi bằng trung vị
trung_vi_tuoi = df_khachhang["Tuoi"].median()
df_khachhang["Tuoi"] = df_khachhang["Tuoi"].fillna(trung_vi_tuoi)

# Điền Giới tính bằng giá trị xuất hiện nhiều nhất
mode_gioi_tinh = df_khachhang["GioiTinh"].mode()[0]
df_khachhang["GioiTinh"] = df_khachhang["GioiTinh"].fillna(mode_gioi_tinh)

print("\nSố lượng giá trị khuyết sau khi xử lý:")
print(df_khachhang.isnull().sum())


### Câu 3: Mã hóa biến phân loại

Cột `ThanhPho` không có thứ tự nên sử dụng One-Hot Encoding. Vẫn giữ lại cột gốc để dùng cho các câu sau.


In [ ]:
# One-Hot Encoding cho cột Thành phố
thanh_pho_encoded = pd.get_dummies(
    df_khachhang["ThanhPho"],
    prefix="ThanhPho",
    dtype=int
)

# Ghép các cột mới vào DataFrame
df_khachhang = pd.concat(
    [df_khachhang, thanh_pho_encoded],
    axis=1
)

df_khachhang.head()


### Câu 4: Phát hiện và xử lý điểm dị biệt bằng IQR

In [ ]:
# Tính Q1, Q3 và IQR của cột ThuNhap
Q1 = df_khachhang["ThuNhap"].quantile(0.25)
Q3 = df_khachhang["ThuNhap"].quantile(0.75)
IQR = Q3 - Q1

gioi_han_duoi = Q1 - 1.5 * IQR
gioi_han_tren = Q3 + 1.5 * IQR

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Giới hạn dưới:", gioi_han_duoi)
print("Giới hạn trên:", gioi_han_tren)

# Lấy các dòng bị xem là outlier
outliers = df_khachhang[
    (df_khachhang["ThuNhap"] < gioi_han_duoi) |
    (df_khachhang["ThuNhap"] > gioi_han_tren)
]

print("\nSố dòng outlier:", len(outliers))

# Xóa các dòng có ThuNhap là outlier
df_khachhang = df_khachhang[
    (df_khachhang["ThuNhap"] >= gioi_han_duoi) &
    (df_khachhang["ThuNhap"] <= gioi_han_tren)
].copy()

df_khachhang.reset_index(drop=True, inplace=True)

print("Số dòng còn lại:", len(df_khachhang))


### Câu 5: Chuẩn hóa dữ liệu bằng MinMaxScaler

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df_khachhang["TongChiTieu_Scaled"] = scaler.fit_transform(
    df_khachhang[["TongChiTieu"]]
)

df_khachhang[["TongChiTieu", "TongChiTieu_Scaled"]].head()


### Câu 6: Lọc dữ liệu theo điều kiện

In [ ]:
# Khách hàng Nữ, tuổi trên 30 và ở Hà Nội
df_phu = df_khachhang[
    (df_khachhang["GioiTinh"] == "Nữ") &
    (df_khachhang["Tuoi"] > 30) &
    (df_khachhang["ThanhPho"] == "Hà Nội")
]

df_phu.head()


### Câu 7: Gom nhóm và thống kê

In [ ]:
# Tính trung bình và tổng chi tiêu theo từng thành phố
thong_ke_thanh_pho = df_khachhang.groupby("ThanhPho")["TongChiTieu"].agg(
    TrungBinh="mean",
    Tong="sum"
)

thong_ke_thanh_pho


### Câu 8: Kỹ nghệ đặc trưng

In [ ]:
# Chia nhóm tuổi bằng pd.cut
moc_tuoi = [17, 30, 45, 60, np.inf]
ten_nhom = ["18-30", "31-45", "46-60", "Trên 60"]

df_khachhang["NhomTuoi"] = pd.cut(
    df_khachhang["Tuoi"],
    bins=moc_tuoi,
    labels=ten_nhom
)

df_khachhang[["Tuoi", "NhomTuoi"]].head(10)


### Câu 9: Ma trận tương quan Pearson

In [ ]:
# Tính ma trận tương quan
ma_tran_tuong_quan = df_khachhang[
    ["Tuoi", "ThuNhap", "TongChiTieu"]
].corr(method="pearson")

print(ma_tran_tuong_quan)

# Vẽ heatmap
plt.figure(figsize=(7, 5))
sns.heatmap(
    ma_tran_tuong_quan,
    annot=True,
    fmt=".2f",
    cmap="Blues"
)
plt.title("Ma trận tương quan Pearson")
plt.show()


**Nhận xét:** `ThuNhap` và `TongChiTieu` có tương quan dương vì dữ liệu tổng chi tiêu được tạo dựa một phần vào thu nhập. Tuổi được tạo ngẫu nhiên nên mức tương quan thường không cao.


### Câu 10: Trực quan hóa dữ liệu

In [ ]:
plt.figure(figsize=(9, 6))

sns.scatterplot(
    data=df_khachhang,
    x="ThuNhap",
    y="TongChiTieu",
    hue="GioiTinh",
    alpha=0.7
)

plt.title("Mối quan hệ giữa Thu nhập và Tổng chi tiêu")
plt.xlabel("Thu nhập")
plt.ylabel("Tổng chi tiêu")
plt.ticklabel_format(style="plain", axis="both")
plt.show()


## PHẦN 2: ỨNG DỤNG HỌC SÂU - MẠNG CNN

### Câu 11: Xây dựng và huấn luyện mô hình CNN với Fashion MNIST

Fashion MNIST gồm ảnh quần áo kích thước 28x28 pixel và có 10 lớp.


In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

# Tải dữ liệu Fashion MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

print("Kích thước tập train:", x_train.shape)
print("Kích thước tập test:", x_test.shape)


In [ ]:
# Chuẩn hóa pixel về khoảng [0, 1]
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Thêm chiều kênh màu: (28, 28) thành (28, 28, 1)
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

print("Sau khi reshape:", x_train.shape)


In [ ]:
# Xây dựng mô hình CNN
model = Sequential([
    Conv2D(
        filters=32,
        kernel_size=(3, 3),
        activation="relu",
        input_shape=(28, 28, 1)
    ),
    MaxPooling2D(pool_size=(2, 2)),
    Flatten(),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax")
])

model.summary()


In [ ]:
# Biên dịch mô hình
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# Huấn luyện mô hình trong 5 epochs
history = model.fit(
    x_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.1
)


In [ ]:
# Đánh giá mô hình trên tập kiểm thử
test_loss, test_accuracy = model.evaluate(
    x_test,
    y_test,
    verbose=0
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)


**Kết luận:** Mô hình CNN đơn giản đã được huấn luyện trên Fashion MNIST. Độ chính xác thực tế có thể thay đổi nhẹ giữa các lần chạy do quá trình khởi tạo trọng số và huấn luyện.
